In [206]:
import pandas as pd
import datetime

In [207]:
data = pd.read_csv('LECLERC_cleaned.csv')

In [208]:
def calculate_days(row):
    date_range = row['Delivery Date']
    date_range = date_range.replace('Prévue entre le ', '')
    date_range = date_range.replace('Prévue le ', '')
    if ' et le ' in date_range:
        max_date = date_range.split(' et le ')[-1]
    else:
        max_date = date_range
    max_date_dt = datetime.datetime.strptime(max_date, '%d/%m/%y')
    scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
    scrap_date = datetime.datetime.combine(scrap_date.date(), datetime.time(0, 0))
    
    difference = (max_date_dt - scrap_date).days
    
    return difference

In [ ]:
import numpy as np

def calculate_experience(row):
    # Correct way to check for np.nan
    if not pd.isna(row['SellerActivityDate']):
        start = str(row['SellerActivityDate'])
        start_dt = datetime.datetime.strptime(start, '%d/%m/%Y')
        scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
        
        scrap_date = scrap_date.date()
        start_dt = start_dt.date()
        
        difference = (scrap_date - start_dt).days
    else:
        difference = np.nan
    
    return difference

In [223]:
calculate_experience({"SellerActivityDate":'23/12/2024',
                        "Timestamp":'12/O1/2026 10:56:02'
                      })

1

In [210]:
data['shipping_days'] = data.apply(calculate_days, axis=1)

Ajout des infos vendeurs 

In [211]:
data.drop(columns = ['Seller Rating','Delivery Date','Platform'],inplace=True)

In [212]:
seller = pd.read_csv('Sellers.csv')
seller.drop(columns = ['SellerStatus'],inplace=True)

In [213]:
data_merge = pd.merge(data,seller,how='left',on='Seller')

In [214]:
data_merge.head()

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerActivityDate,SellerCountry
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.00,Offerte,NEUF,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,NaN,France
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Icoza,1408.17,Offerte,NEUF,24/12/2024 13:30:11,98b12821-acb5-4aa6-a5d4-129db155e1d9,4,4.0,140.0,11/09/2020,France
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Stock e-commerce,1409.17,Offerte,NEUF,24/12/2024 13:30:11,5ace8dfe-bef8-428f-9a20-eda099c4661a,4,4.0,242.0,17/06/2021,France
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Maxmovil,1432.28,Offerte,NEUF,24/12/2024 13:30:11,5e16f708-0b47-4d88-9e49-ef4d3dd29796,4,4.0,14.0,25/06/2021,Espagne
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Monsieurplus,1473.99,Offerte,NEUF,24/12/2024 13:30:11,d282bc92-68ff-4039-9239-32e2443a6a19,4,4.0,7.0,20/12/2022,France


In [215]:
data_merge.isna().sum()/len(data_merge)

Product Name          0.000000
Seller                0.000000
Price                 0.000000
Delivery Fees         0.000000
Product State         0.000000
Timestamp             0.000000
ID                    0.000000
shipping_days         0.000000
Seller Rating         0.109126
NbSellerRatings       0.109126
SellerActivityDate    0.109126
SellerCountry         0.000000
dtype: float64

In [216]:
data_merge[data_merge['SellerActivityDate'].apply(lambda x: isinstance(x, float))].head(3)

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerActivityDate,SellerCountry
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.0,Offerte,NEUF,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,NaN,France
9,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1098.0,Offerte,NEUF,24/12/2024 13:30:17,3daccedd-2936-4a16-bb57-abf5e372681c,4,NaN,NaN,NaN,France
19,Non trouvé,E.Leclerc,969.0,Offerte,Non trouvé,24/12/2024 13:30:24,b22e1674-9148-4c9c-8afa-071b9cf27bbf,4,NaN,NaN,NaN,France


In [193]:
data_merge['SellerExperience'] = data_merge.apply(calculate_experience, axis=1)

ValueError: unconverted data remains: 20

In [ ]:
data_merge.isna().sum()

Product Name               0
Seller                     0
Price                      0
Delivery Fees              0
Product State              0
Timestamp                  0
ID                         0
shipping_days              0
Seller Rating          22406
NbSellerRatings        22406
SellerActivityDate     22406
SellerCountry              0
SellerExperience      205323
dtype: int64

Suppression de delivery fees car sont tous = offerte

In [ ]:
columns_to_delete = ['Delivery Fees','SellerActivityDate','Seller','Product State']

In [ ]:
#Supprimer la catégorie d'état de téléphone la plus fréquente comme variable de reférence
len(data_merge[data_merge['Product State'] == 'NEUF']['ID'].unique())/len(data_merge['ID'].unique())

0.910958904109589

In [ ]:
dummies = pd.get_dummies(data_merge['Product State'], drop_first=True)
data_merge = pd.concat([data_merge, dummies], axis=1)

In [ ]:
data_merge.head()

,Product Name,Price,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerCountry,SellerExperience,Non trouvé,OCCASION - BON ÉTAT,OCCASION - EXCELLENT ÉTAT,OCCASION - PARFAIT - JAMAIS UTILISÉ,OCCASION - TRÉS BON ÉTAT,OCCASION - ÉTAT CORRECT
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",1349.00,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,France,NaN,False,False,False,False,False,False
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",1408.17,24/12/2024 13:30:11,98b12821-acb5-4aa6-a5d4-129db155e1d9,4,4.0,140.0,France,NaN,False,False,False,False,False,False
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",1409.17,24/12/2024 13:30:11,5ace8dfe-bef8-428f-9a20-eda099c4661a,4,4.0,242.0,France,NaN,False,False,False,False,False,False
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",1432.28,24/12/2024 13:30:11,5e16f708-0b47-4d88-9e49-ef4d3dd29796,4,4.0,14.0,Espagne,NaN,False,False,False,False,False,False
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",1473.99,24/12/2024 13:30:11,d282bc92-68ff-4039-9239-32e2443a6a19,4,4.0,7.0,France,NaN,False,False,False,False,False,False


In [ ]:
data_merge.drop(columns = columns_to_delete,inplace=True)

In [ ]:
data_merge.isna().sum()

Product Name                                0
Price                                       0
Timestamp                                   0
ID                                          0
shipping_days                               0
Seller Rating                           22406
NbSellerRatings                         22406
SellerCountry                               0
SellerExperience                       205323
Non trouvé                                  0
OCCASION - BON ÉTAT                         0
OCCASION - EXCELLENT ÉTAT                   0
OCCASION - PARFAIT - JAMAIS UTILISÉ         0
OCCASION - TRÉS BON ÉTAT                    0
OCCASION - ÉTAT CORRECT                     0
dtype: int64